# Verdade ou Fake? — app no Google Colab
Execute as células em ordem (Ambiente de execução → Executar tudo). A URL pública
aparece no final da última célula e **muda a cada execução**. Suba a sessão ~20 min
antes da apresentação e não a feche.

In [ ]:
REPO = "https://github.com/unb-Sistemas-de-Machine-learning/challenge_1_megatron.git"
BRANCH = "main"  # o card bertimbau.json precisa estar em status "producao" nesta branch
MODELO = "Soneka39/bertimbau-saude"
REVISAO = "cca4ccd4f969db10df72fa28f0771f97771ae255"
NCBI_EMAIL = ""  # opcional: e-mail de contato pedido pelo NCBI

In [ ]:
import subprocess
subprocess.run(["rm", "-rf", "/content/app"], check=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/app"], check=True)
import json
status = json.load(open("/content/app/modelos/cards/bertimbau.json"))["status"]
assert status == "producao", f"Card em '{status}': faça merge da promoção na {BRANCH} (ou ajuste BRANCH)."
print("card ok:", status)

In [ ]:
# Mantém o torch do Colab e instala o resto das dependências fixadas.
!grep -v '^torch' /content/app/requirements.txt > /tmp/req.txt
!pip install -q -r /tmp/req.txt

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, subprocess, time, re, urllib.request
os.environ.update(VOF_MODELO_RISCO=MODELO, VOF_MODELO_RISCO_REVISAO=REVISAO, NCBI_EMAIL=NCBI_EMAIL)

app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.headless=true",
     "--browser.gatherUsageStats=false"],
    cwd="/content/app", stdout=open("/tmp/streamlit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:8501/_stcore/health"); break
    except Exception:
        time.sleep(2)

tunel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in tunel.stdout:
    achou = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", linha)
    if achou:
        print("\nURL pública:", achou.group(0)); break
print("O primeiro acesso baixa os modelos (alguns minutos). Mantenha esta aba aberta.")